In [1]:
# -*- coding: utf-8 -*-

# Phase 2 — EDA와 유연성 프로파일링 (v4)

`out/phase1_분석테이블.csv`를 입력으로 **무엇이 유연성을 좌우하는가**를 기술통계로 확인한다.
모델링(Phase 4·5) 전에 반드시 눈으로 먼저 본다.

| 절 | 내용 | 판단 기준 |
|---|---|---|
| 2-1 | 단변량 분포 | 시간 고정(0분) 비율 = 시장 경직성 |
| 2-2 | 교차분석 | Cramér's V ≥ 0.3 → 품목별 차등 제안 |
| 2-3 | 등록주체 비교 | 원화주 승인링크의 근거 검정 |
| 2-4 | **조건 → 결과** | v4 신규. 조건이 실제로 운임·유찰을 만드는가 |

**v3 대비 변경**

- `자동조정_허용`·`최소절감_임계_원` 소멸 → `시간변경비용_원_시간`으로 대체
- `도착_시도` → `도착권역`, `알선소_관할구`(19개) → `알선소_관할권역`(4개)
- **품목 × 적재형태를 새로 넣었다.** v2는 이 관계가 1:1(설명력 1.000)이라 검정이 무의미했는데
  v4는 확률 배분(0.444)이라 비로소 "품목이 차종을 얼마나 규정하는가"를 물을 수 있다.
- 2-4절 신설. v2에는 결과 필드가 없어 존재할 수 없던 분석이다.

In [2]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind, kruskal, binomtest
from scipy.stats import t as _t
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 한글 폰트 — 없는 환경에서도 죽지 않게 후보를 순회한다
for _f in ["Noto Sans CJK KR", "Noto Sans CJK JP", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
    if any(_f in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False

load_dotenv()
SRC = Path(os.getenv("DATA_DIR", ".")).expanduser().resolve()
OUT = SRC / "out"; OUT.mkdir(exist_ok=True)
FIG = OUT / "fig"; FIG.mkdir(exist_ok=True)

df = pd.read_csv(OUT / "phase1_분석테이블.csv")
n = len(df)
print(f"[load] {n}행 / 화주 {df['화주ID'].nunique()}명")


def holm(pvals):
    """Holm-Bonferroni 다중비교 보정"""
    p = np.asarray(pvals, float); m = len(p)
    idx = np.argsort(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[idx])
    out = np.empty(m); out[idx] = np.clip(adj, 0, 1)
    return out


def cramers_v(tab):
    """Cramér's V + Bergsma(2013) 편향보정"""
    chi2, p, dof, exp = chi2_contingency(tab)
    n_ = tab.values.sum(); r, c = tab.shape
    v = np.sqrt(chi2 / (n_ * min(r - 1, c - 1)))
    phi2 = chi2 / n_
    phi2c = max(0, phi2 - (r - 1) * (c - 1) / (n_ - 1))
    rc = r - (r - 1) ** 2 / (n_ - 1); cc = c - (c - 1) ** 2 / (n_ - 1)
    vc = np.sqrt(phi2c / max(min(rc - 1, cc - 1), 1e-12))
    return dict(chi2=chi2, p=p, dof=dof, V=v, V_corrected=vc,
                기대빈도5미만_셀=int((exp < 5).sum()), 전체셀=exp.size, n=int(n_))


def interp(v):
    return "강함(≥0.5)" if v >= .5 else "실질적(≥0.3)" if v >= .3 else "약함(≥0.1)" if v >= .1 else "미미"

[load] 12000행 / 화주 300명


## 2-1. 단변량 분포 — 고정 비율이 곧 시장의 경직성

In [3]:
print("\n" + "=" * 60 + "\n2-1. 단변량 분포\n" + "=" * 60)

rng = df["시간창_분"].value_counts().sort_index()
uni_rows = [("시간창_분", str(k), v, round(v / n * 100, 1)) for k, v in rng.items()]

for col, pos in [("차량유연성", "대체허용"), ("분할운송", "Y"), ("동시적재", "Y"),
                 ("경유허용", "Y"), ("상하차_순서변경", "Y")]:
    c = int((df[col] == pos).sum())
    uni_rows.append((col, pos, c, round(c / n * 100, 1)))

for k, v in df["원화주_조정권한_증빙"].fillna("원화주직접").value_counts().items():
    uni_rows.append(("원화주_조정권한_증빙", str(k), v, round(v / n * 100, 1)))

cost = df.loc[df["시간변경비용_원_시간"] > 0, "시간변경비용_원_시간"].value_counts().sort_index()
for k, v in cost.items():
    uni_rows.append(("시간변경비용_원_시간(>0)", f"{int(k):,}", v, round(v / cost.sum() * 100, 1)))

pd.DataFrame(uni_rows, columns=["변수", "값", "건수", "비율%"]).to_csv(
    OUT / "phase2_단변량분포.csv", index=False, encoding="utf-8-sig")

fixed_rate = (df["시간창_분"] == 0).mean() * 100
print(f"  시간 고정(0분) 비율        : {fixed_rate:.1f}%  ← 시장 경직성 지표")
print(f"  하루 이상 개방(≥1440분)    : {(df['시간창_분'] >= 1440).mean()*100:.1f}%")
print(f"  차량 대체허용률            : {(df['차량유연성']=='대체허용').mean()*100:.1f}%")
print(f"  분할운송 허용률            : {(df['분할운송']=='Y').mean()*100:.1f}%")
print(f"  시간변경비용 0원(무비용) 비율: {(df['시간변경비용_원_시간']==0).mean()*100:.1f}%")
print(f"  유연성지수 중앙값          : {df['유연성지수'].median():.3f}")


2-1. 단변량 분포
  시간 고정(0분) 비율        : 38.2%  ← 시장 경직성 지표
  하루 이상 개방(≥1440분)    : 7.3%
  차량 대체허용률            : 21.7%
  분할운송 허용률            : 10.8%
  시간변경비용 0원(무비용) 비율: 24.4%
  유연성지수 중앙값          : 0.325


## 2-2. 교차분석 — 유연성의 구조적 결정요인

표본이 900 → 4,000으로 늘어 검정력이 올라갔다. 그만큼 **p값은 쉽게 유의해지므로
판단은 p가 아니라 효과크기(V)로 한다.**

In [4]:
print("\n" + "=" * 60 + "\n2-2. 교차분석 (카이제곱 · Cramér's V)\n" + "=" * 60)

pairs = [
    ("품목", "시간창_분"),
    ("품목", "적재형태"),        # v4 신규 — v2는 1:1이라 검정 불가였다
    ("품목", "차량유연성"),
    ("품목", "분할운송"),
    ("차종", "시간창_분"),
    ("긴급여부", "시간창_분"),
    ("긴급여부", "결과"),
    ("도착권역", "시간창_분"),
    ("등록주체", "시간창_분"),
    ("등록주체", "원화주_조정권한_증빙"),
    ("알선소_구분", "원화주_조정권한_증빙"),
    ("시간창_분", "결과"),        # 핵심 — 조건이 결과를 만드는가
]
res = []
for a, b in pairs:
    tab = pd.crosstab(df[a].fillna("없음"), df[b].fillna("없음"))
    r = cramers_v(tab)
    r.update({"행": a, "열": b, "해석": interp(r["V"])})
    res.append(r)

cross = pd.DataFrame(res)[["행", "열", "n", "chi2", "dof", "p", "V", "V_corrected",
                           "해석", "기대빈도5미만_셀", "전체셀"]].sort_values("V", ascending=False)
cross["p_holm"] = holm(cross["p"].values)
cross["유의(α=.05,보정후)"] = np.where(cross["p_holm"] < .05, "O", "X")
cross.round(4).to_csv(OUT / "phase2_교차검정.csv", index=False, encoding="utf-8-sig")
print(cross[["행", "열", "chi2", "p", "V", "해석", "기대빈도5미만_셀", "전체셀"]].round(3).to_string(index=False))

# 핵심 교차표: 품목 × 시간창 (행 정규화)
tab_item = pd.crosstab(df["품목"], df["시간창_분"], normalize="index").round(3)
tab_item = tab_item.loc[df.groupby("품목")["유연성지수"].mean().sort_values(ascending=False).index]
tab_item.to_csv(OUT / "phase2_교차표_품목x시간창.csv", encoding="utf-8-sig")
print("\n[품목 × 시간창_분] 행 정규화")
print(tab_item.to_string())

H, p_kw = kruskal(*[g["유연성지수"].values for _, g in df.groupby("품목")])
k_ = df["품목"].nunique()
print(f"\n[Kruskal-Wallis] 품목 간 유연성지수 차이: H={H:.1f}, p={p_kw:.3e}, "
      f"ε²={(H - k_ + 1) / (n - k_):.3f}")

item_summary = (df.groupby("품목")
                .agg(건수=("유연성지수", "size"), 평균유연성=("유연성지수", "mean"),
                     시간고정률=("시간창_분", lambda s: (s == 0).mean()),
                     대체허용률=("차량유연성", lambda s: (s == "대체허용").mean()),
                     긴급비율=("긴급여부", lambda s: (s == "긴급").mean()),
                     유찰률=("유찰", "mean"), 체결배율=("체결배율", "mean"))
                .sort_values("평균유연성", ascending=False).round(3))
item_summary.to_csv(OUT / "phase2_품목프로파일.csv", encoding="utf-8-sig")
print("\n[품목 프로파일]"); print(item_summary.to_string())

# ── 강건성 점검 ────────────────────────────────────────────────
# (a) 긴급 교란 배제 — 긴급건을 빼도 품목 효과가 남는가
_ng = df[df["긴급여부"] == "일반"]
r_ng = cramers_v(pd.crosstab(_ng["품목"], _ng["시간창_분"]))
print(f"\n[강건성] 일반건만(n={len(_ng)}): V={r_ng['V']:.3f}, p={r_ng['p']:.2e}"
      "  → 품목 효과는 긴급 교란의 산물이 아님")

# (b) 권한 게이트 배제 — 미승인은 시간창이 0으로 강제되므로 인위적 연관을 만든다
_ok = df[df["권한_미승인"] == 0]
r_ok = cramers_v(pd.crosstab(_ok["품목"], _ok["시간창_분"]))
print(f"[강건성] 권한 정상건만(n={len(_ok)}): V={r_ok['V']:.3f}, p={r_ok['p']:.2e}"
      "  → 게이트가 만든 가짜 연관이 아닌지 확인")

# (c) 온도군 2분류로 축약해도 실질적 연관이 유지되는가
COLD = ["냉동수산", "냉동식품", "제과류", "식품가공"]
df["온도군"] = np.where(df["품목"].isin(COLD), "냉장·식품", "상온·공산")
r_temp = cramers_v(pd.crosstab(df["온도군"], df["시간창_분"] == 0))
print(f"[강건성] 온도군 2분류 × 시간고정: phi={r_temp['V']:.3f}, p={r_temp['p']:.2e}")
print(df.groupby("온도군")["유연성지수"].agg(["count", "mean"]).round(3).to_string())


2-2. 교차분석 (카이제곱 · Cramér's V)
     행           열      chi2     p     V       해석  기대빈도5미만_셀  전체셀
  등록주체 원화주_조정권한_증빙 12000.000 0.000 1.000 강함(≥0.5)          0    8
    품목        적재형태 25914.078 0.000 0.735 강함(≥0.5)          0   55
    품목       시간창_분   717.462 0.000 0.109 약함(≥0.1)          0   66
  등록주체       시간창_분    97.836 0.000 0.090       미미          0   12
  도착권역       시간창_분   222.638 0.000 0.079       미미          0   24
 시간창_분          결과    65.236 0.000 0.074       미미          0   12
    품목       차량유연성    43.360 0.000 0.060       미미          0   22
  긴급여부       시간창_분    33.625 0.000 0.053       미미          0   12
    차종       시간창_분   167.413 0.000 0.053       미미          5   90
  긴급여부          결과    22.152 0.000 0.043       미미          0    4
알선소_구분 원화주_조정권한_증빙    11.219 0.011 0.031       미미          0    8
    품목        분할운송     6.564 0.766 0.023       미미          0   22

[품목 × 시간창_분] 행 정규화
시간창_분   0      30     60     120    240    2880
품목                                         

## 2-3. 등록주체별 비교 — 원화주 승인링크의 근거

"주선사가 대리 등록하면 유연성이 줄어드는가"를 검정한다.
유의차가 없을 때 **검정력 부족인지 실제 동등인지**를 TOST 등가성 검정으로 가른다.

> **⚠ 지표 선택 주의.** `유연성지수`에는 `flex_auth`가 들어 있고, 이 값은
> 등록주체가 원화주직접이면 정의상 1.0이다. 그대로 쓰면 "원화주직접이 더 유연하다"는
> 결론이 산술적으로 보장된다 — 순환 논증이다.
> 그래서 **auth를 뺀 `유연성지수_코어`로 검정**하고, 오염된 원지표 결과도 대조용으로 같이 낸다.

In [5]:
print("\n" + "=" * 60 + "\n2-3. 등록주체별 비교\n" + "=" * 60)

_raw_a = df.loc[df["등록주체"] == "주선사대리", "유연성지수"]
_raw_b = df.loc[df["등록주체"] == "원화주직접", "유연성지수"]
print(f"  [대조] 원지표(auth 포함) 평균차 = {_raw_a.mean() - _raw_b.mean():+.3f}"
      "  ← flex_auth 때문에 구조적으로 벌어지는 값. 근거로 쓰면 안 된다")

a = df.loc[df["등록주체"] == "주선사대리", "유연성지수_코어"]
b = df.loc[df["등록주체"] == "원화주직접", "유연성지수_코어"]
t, p_t = ttest_ind(a, b, equal_var=False)
u, p_u = mannwhitneyu(a, b, alternative="two-sided")
pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
d = (a.mean() - b.mean()) / pooled
rb = 2 * u / (len(a) * len(b)) - 1
print(f"  주선사대리 n={len(a)} mean={a.mean():.3f} / 원화주직접 n={len(b)} mean={b.mean():.3f}")
print(f"  Welch t={t:.3f}, p={p_t:.4f} | Mann-Whitney U p={p_u:.4f}")
print(f"  Cohen's d={d:.3f} (|d|<0.2 = 무시 가능) | rank-biserial={rb:.3f}")

delta = 0.2 * pooled
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
dfree = (a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)) ** 2 / (
    (a.var(ddof=1) / len(a)) ** 2 / (len(a) - 1) + (b.var(ddof=1) / len(b)) ** 2 / (len(b) - 1))
diff = a.mean() - b.mean()
p_tost = max(1 - _t.cdf((diff + delta) / se, dfree), _t.cdf((diff - delta) / se, dfree))
print(f"  TOST 등가성 p={p_tost:.4f} (등가한계 ±{delta:.4f})")

if p_u < .05 and d < -0.2:
    verdict = "가설 지지 — 대리등록이 유연성을 축소"
elif p_tost < .05:
    verdict = "가설 기각(등가 입증) — 등록주체 자체는 유연성과 무관"
else:
    verdict = "가설 미지지 — 유의차 없으나 등가도 입증 못함(판단 보류)"
print(f"  → {verdict}")

pd.DataFrame([{
    "비교": "주선사대리 vs 원화주직접(유연성지수)", "n_대리": len(a), "n_직접": len(b),
    "mean_대리": round(a.mean(), 4), "mean_직접": round(b.mean(), 4),
    "welch_t": round(t, 4), "p_ttest": round(p_t, 4),
    "mannwhitney_U": float(u), "p_mwu": round(p_u, 4),
    "cohens_d": round(d, 4), "rank_biserial": round(rb, 4), "판정": verdict,
}]).to_csv(OUT / "phase2_등록주체검정.csv", index=False, encoding="utf-8-sig")

# ── 메커니즘 분리 ──────────────────────────────────────────────
# 대리등록이 유연성을 줄이는 게 '대리' 자체 때문인가, 아니면 그중 권한 미승인·대기 건이
# 유연성 필드를 잠그기 때문인가. 승인완료 건만 남기고 다시 비교하면 갈린다.
_ok_deleg = df.loc[df["원화주_조정권한_증빙"] == "승인완료", "유연성지수_코어"]
_t2, _p2 = ttest_ind(_ok_deleg, b, equal_var=False)
_pool2 = np.sqrt(((len(_ok_deleg) - 1) * _ok_deleg.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                 / (len(_ok_deleg) + len(b) - 2))
_d2 = (_ok_deleg.mean() - b.mean()) / _pool2
print(f"\n  [메커니즘] 승인완료 대리(n={len(_ok_deleg)}, {_ok_deleg.mean():.3f}) "
      f"vs 원화주직접(n={len(b)}, {b.mean():.3f}) → d={_d2:.3f}, p={_p2:.4f}")
print("     |d|가 위의 -0.286보다 크게 줄면 원인은 '대리'가 아니라 '권한 게이트'다"
      " → 해법은 대리등록 차단이 아니라 원화주 승인링크")

# 진짜 문제는 등록주체가 아니라 권한 증빙이다
print("\n[원화주 조정권한 증빙]")
auth = df["원화주_조정권한_증빙"].fillna("원화주직접").value_counts()
print(auth.to_frame("건수").assign(비율=lambda x: (x["건수"] / n * 100).round(1)).to_string())

deleg = df[df["등록주체"] == "주선사대리"].copy()
tab_auth = pd.crosstab(deleg["알선소_구분"], deleg["원화주_조정권한_증빙"])
r_auth = cramers_v(tab_auth)
print(f"\n[대리등록 {len(deleg)}건] 미승인 {deleg['권한_미승인'].sum()}건 "
      f"({deleg['권한_미승인'].mean()*100:.1f}%)")
print(tab_auth.to_string())
print(f"  주선사/운송사 × 증빙상태: chi2={r_auth['chi2']:.2f}, p={r_auth['p']:.4f}, V={r_auth['V']:.3f}")

base_rate = deleg["권한_미승인"].mean()
gu = (deleg.groupby("알선소_관할권역")
      .agg(대리건수=("권한_미승인", "size"), 미승인=("권한_미승인", "sum"))
      .query("대리건수 >= 30"))
gu["미승인률"] = (gu["미승인"] / gu["대리건수"]).round(3)
gu["p_binom"] = [binomtest(int(s_), int(n_), base_rate).pvalue
                 for n_, s_ in zip(gu["대리건수"], gu["미승인"])]
gu["p_holm"] = holm(gu["p_binom"].values)
gu["기준초과_유의"] = np.where(gu["p_holm"] < .05, "O", "X")
gu = gu.sort_values("미승인률", ascending=False)
gu.round(4).to_csv(OUT / "phase2_권역별_미승인률.csv", encoding="utf-8-sig")
print(f"\n[관할권역별 미승인률] 기준(대리 전체)={base_rate:.3f}")
print(gu.round(4).to_string())
print(f"  Holm 보정 후 기준 초과 유의: {int((gu['p_holm']<.05).sum())}개 / {len(gu)}개"
      "  → 0개면 '고위험 권역' 주장 불가(표본변동)")


2-3. 등록주체별 비교
  [대조] 원지표(auth 포함) 평균차 = -0.111  ← flex_auth 때문에 구조적으로 벌어지는 값. 근거로 쓰면 안 된다
  주선사대리 n=9415 mean=0.223 / 원화주직접 n=2585 mean=0.287
  Welch t=-14.128, p=0.0000 | Mann-Whitney U p=0.0000
  Cohen's d=-0.311 (|d|<0.2 = 무시 가능) | rank-biserial=-0.199
  TOST 등가성 p=1.0000 (등가한계 ±0.0414)
  → 가설 지지 — 대리등록이 유연성을 축소

  [메커니즘] 승인완료 대리(n=5306, 0.286) vs 원화주직접(n=2585, 0.287) → d=-0.004, p=0.8580
     |d|가 위의 -0.286보다 크게 줄면 원인은 '대리'가 아니라 '권한 게이트'다 → 해법은 대리등록 차단이 아니라 원화주 승인링크

[원화주 조정권한 증빙]
               건수    비율
원화주_조정권한_증빙            
승인완료         5306  44.2
승인대기         2598  21.6
원화주직접        2585  21.5
미승인          1511  12.6

[대리등록 9415건] 미승인 1511건 (16.0%)
원화주_조정권한_증빙  미승인  승인대기  승인완료
알선소_구분                      
운송사          849  1467  3142
주선사          662  1131  2164
  주선사/운송사 × 증빙상태: chi2=7.76, p=0.0206, V=0.029

[관할권역별 미승인률] 기준(대리 전체)=0.160
          대리건수  미승인   미승인률  p_binom  p_holm 기준초과_유의
알선소_관할권역                                           
충청         792  137  0.173   0.3331 

## 2-4. 조건 → 결과 (v4 신규)

**v2에는 결과 필드가 없어 존재할 수 없던 절이다.** 여기가 제품의 핵심 주장을 데이터로
직접 확인하는 자리다 — "조건을 좁게 걸면 운임이 오르고 유찰이 난다".

주의: 이건 인과 추정이 아니라 기술통계다. 조건과 결과 사이에 화주 성향·품목 같은
교란이 있으므로, 인과에 가까운 추정은 Phase 4에서 공변량을 통제한 뒤 한다.

In [6]:
print("\n" + "=" * 60 + "\n2-4. 조건 → 결과\n" + "=" * 60)

df["창구간"] = pd.cut(df["시간창_분"], [-1, 0, 60, 240, 1439, 1e9],
                    labels=["고정(0분)", "1시간 이하", "4시간 이하", "하루 미만", "하루 이상"])
by_win = (df.groupby("창구간", observed=True)
          .agg(건수=("콜ID", "size"), 유찰률=("유찰", "mean"),
               체결배율=("체결배율", "mean"), 인상횟수=("인상횟수", "mean"),
               배차분=("배차소요_분", "mean"), 후보수=("수락가능_현조건", "median"))
          .round(3))
by_win.to_csv(OUT / "phase2_시간창별_결과.csv", encoding="utf-8-sig")
print("[시간창 구간별 결과]"); print(by_win.to_string())

by_urg = (df.groupby("긴급여부")
          .agg(건수=("콜ID", "size"), 유찰률=("유찰", "mean"), 체결배율=("체결배율", "mean"),
               배차분=("배차소요_분", "mean"), 초과운임=("초과운임", "mean")).round(3))
print("\n[긴급 여부별 결과]"); print(by_urg.to_string())

# 긴급 건이 초과운임에서 차지하는 비중 — 기획서 '61%' 대체 근거
s1 = df[df["결과"] == "성사"]
share = s1.loc[s1["긴급여부"] == "긴급", "초과운임"].sum() / s1["초과운임"].sum()
cnt = (s1["긴급여부"] == "긴급").mean()
print(f"\n[초과운임 집중도] 긴급 건수비 {cnt:.1%} → 초과운임 비중 {share:.1%} "
      f"(건수 대비 {share/cnt:.1f}배 집중)")

# 후보 수와 결과의 관계 — 경매 메커니즘이 실제로 작동하는지
q = pd.qcut(df["수락가능_현조건"], 5, labels=["최소", "적음", "보통", "많음", "최다"], duplicates="drop")
by_cand = df.groupby(q, observed=True).agg(
    유찰률=("유찰", "mean"), 체결배율=("체결배율", "mean"), 배차분=("배차소요_분", "mean")).round(3)
print("\n[수락가능 차주 수 5분위별 결과]"); print(by_cand.to_string())


2-4. 조건 → 결과
[시간창 구간별 결과]
          건수    유찰률   체결배율   인상횟수      배차분    후보수
창구간                                              
고정(0분)  4584  0.096  1.151  2.452  132.400    6.0
1시간 이하  3619  0.094  1.154  2.544  108.083   11.0
4시간 이하  2915  0.090  1.146  2.431   78.407   27.0
하루 이상    882  0.016  1.094  1.202   29.333  553.0

[긴급 여부별 결과]
         건수    유찰률   체결배율      배차분        초과운임
긴급여부                                          
긴급     1615  0.119  1.396  101.091  126610.863
일반    10385  0.083  1.109  104.266   38052.524

[초과운임 집중도] 긴급 건수비 13.0% → 초과운임 비중 33.2% (건수 대비 2.6배 집중)

[수락가능 차주 수 5분위별 결과]
            유찰률   체결배율      배차분
수락가능_현조건                       
최소        0.091  1.146  165.269
적음        0.088  1.156  122.591
보통        0.098  1.148  101.085
많음        0.100  1.158   78.378
최다        0.057  1.122   44.741


## 시각화

In [7]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

rng.plot(kind="bar", ax=ax[0, 0], color="#3B6EA5")
ax[0, 0].set_title(f"시간창 분포 — 고정(0분) {fixed_rate:.0f}%", fontsize=12)
ax[0, 0].set_xlabel("시간창(분)"); ax[0, 0].set_ylabel("건수")

item_summary["평균유연성"].plot(kind="barh", ax=ax[0, 1], color="#C25E5E")
ax[0, 1].invert_yaxis(); ax[0, 1].set_title("품목별 평균 유연성지수", fontsize=12)

by_win[["유찰률"]].plot(kind="bar", ax=ax[1, 0], color="#C25E5E", legend=False)
ax2 = ax[1, 0].twinx()
by_win["체결배율"].plot(ax=ax2, color="#3B6EA5", marker="o")
ax[1, 0].set_title("시간창 구간별 유찰률(막대) · 체결배율(선)", fontsize=12)
ax[1, 0].set_xlabel("")

top = cross.head(6)
ax[1, 1].barh([f"{r}×{c}" for r, c in zip(top["행"], top["열"])], top["V"], color="#5B8C5A")
ax[1, 1].invert_yaxis(); ax[1, 1].axvline(.3, ls="--", c="red", lw=1)
ax[1, 1].set_title("연관 강도 Cramér's V (빨간선=0.3)", fontsize=12)

plt.tight_layout()
plt.savefig(FIG / "phase2_요약.png", dpi=130, bbox_inches="tight")
print(f"\n[저장] {OUT}")


[저장] /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/out


/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_54148/383440446.py:21: UserWarning: Glyph 233 (\N{LATIN SMALL LETTER E WITH ACUTE}) missing from font(s) AppleGothic.
  plt.tight_layout()
/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_54148/383440446.py:22: UserWarning: Glyph 233 (\N{LATIN SMALL LETTER E WITH ACUTE}) missing from font(s) AppleGothic.
  plt.savefig(FIG / "phase2_요약.png", dpi=130, bbox_inches="tight")
